# Maximum Likelihood  #

The Phat distribution can be fitted to a univariate dataset by Maximum Likelihood Estimation (MLE). This is available through `Phat.fit`, which maximises the likelihood with `scipy.optimize` and reports standard errors from the observed information.

There is one major issue with the tails that must be considered.

First, let's fit the Phat distribution to our familiar series of S&P 500 index returns.

In [1]:
import seaborn as sns; sns.set(style = 'whitegrid')

In [2]:
import pandas as pd
import phat as ph

sp_ret = pd.read_csv('../data/sp500_returns.csv', index_col='date', parse_dates=True)['return']

res = ph.Phat.fit(sp_ret)

In [3]:
res.params_dict

{'mu': 0.000571647274575266,
 'sigma': 0.003271635219148546,
 'xi_left': 0.0,
 'xi_right': 0.0}

In [4]:
print(res.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 64028.5212)
           estimate      std err
mu         0.000572     0.000049
sigma      0.003272     0.000030
xi_left     0.000000     0.007089
xi_right     0.000000     0.009555


We can see that both the left and right tail shapes are much smaller than we have estimated using the [POT](pot_ref.ipynb) and [Hill Double Bootstrap techniques](dblbs.ipynb). There is a structural reason. In Phat the shape $\xi$ of a tail also fixes where its junction sits and how much probability lies beyond it, and the junctions are less than one $\sigma$ from the centre. So $\xi$ has to describe the shoulders of the distribution, where most of the observations are, as well as the extreme tail, and the likelihood is dominated by the shoulders. The result is thinner fitted tails that understate the risk of extreme events.

Instead, we can estimate the tail shapes separately and pass them as fixed values to `fit`. This leaves two free parameters: the location $\mu$ and the scale $\sigma$.

In [5]:
xi_left, xi_right = ph.two_tailed_hill_double_bootstrap(sp_ret, seed=42)
xi_left, xi_right

/Users/spindicate/code/rjs/private/projects/phat-preprint/phat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:02<00:25,  2.89s/it]

 20%|██        | 2/10 [00:03<00:13,  1.64s/it]

 30%|███       | 3/10 [00:04<00:08,  1.25s/it]

 40%|████      | 4/10 [00:05<00:06,  1.06s/it]

... (4 outputs left out)

 90%|█████████ | 9/10 [00:09<00:00,  1.23it/s]

100%|██████████| 10/10 [00:09<00:00,  1.24it/s]

100%|██████████| 10/10 [00:09<00:00,  1.01it/s]

(0.3192120423228642, 0.3279048642691963)

In [6]:
res2 = ph.Phat.fit(sp_ret, xi_left, xi_right)
print(res2.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 63243.4148)
           estimate      std err
mu         0.000591     0.000046
sigma      0.002956     0.000030
xi_left 0.3192120423228642      (fixed)
xi_right 0.3279048642691963      (fixed)


In [7]:
res2.params_dict

{'mu': 0.0005910273352607609, 'sigma': 0.0029558310203372036}

In [8]:
res.dist.std(), res2.dist.std(), sp_ret.std()

(0.010541345649623996, 0.019139025073904452, np.float64(0.009929074858842975))